# Sottobosco Trentino – Modello di idoneità dei funghi

Notebook ricavato da `main.py`, suddiviso in blocchi logici. Il modello stima, per ogni cella di 100 m di bosco della Provincia di Trento, quanto è probabile trovare le specie target (*Boletus*, *Cantharellus*, *Craterellus*) in stagione, e produce la previsione per oggi e i due giorni successivi.

**Flusso generale**
1. Parametri e setup
2. Area di studio e griglia
3. Covariate statiche (terreno, bosco, suolo)
4. Osservazioni: presenze GBIF e pseudo-assenze
5. Covariate meteorologiche (storico, feature, interpolazione)
6. Addestramento e validazione dei quattro modelli
7. Previsione meteo per i tre giorni target
8. Poligoni di idoneità e sentieri SAT
9. Dati GeoJSON e mappa HTML

**Avvertenza:** eseguire le celle in ordine. Servono connessione internet e il file di cache meteo `fonti esterne statiche/storico meteo/meteo_2017_2024_storico.npz`. La cella di addestramento Bayes (PyMC) è la più lenta.

## 0. Parametri globali
Costanti che governano l'intero modello: risoluzione della griglia (100 m), sistema di riferimento (EPSG:25832, UTM 32N), specie target, finestre temporali (stagione giugno–novembre, anni minimi per meteo e GBIF), passo della griglia meteo (0.2°), gradiente termico verticale (−6.5 °C per km), lisciatura spaziale delle mappe e percorsi delle sorgenti forestali. Il seed fissa la casualità per rendere i risultati riproducibili.

In [ ]:
# ===================================================================
#  MAPPA FUNGHI TRENTINO – 4 MODELLI + UMIDITÀ/VPD + SAT + GPS
# ===================================================================

RISOLUZIONE    = 100
CRS            = "EPSG:25832"

SPECIE         = [
    "Boletus edulis", "Boletus pinophilus", "Boletus aereus", "Boletus reticulatus",
    "Cantharellus cibarius", "Craterellus tubaeformis"
]

ANNO_MIN_METEO = 2025
ANNO_MIN_GBIF = 2017
MESI_STAGIONE  = (6, 11)
MAX_SFONDO     = 8000
PASSO_METEO    = 0.2
LAPSE          = -0.0065
SMOOTH_M       = 150
MIN_AREA_KM2   = 0.01

TIPI_FORESTALI = "https://siatservices.provincia.tn.it/idt/vector/p_TN_04856367-56db-4f44-bb06-09bf75ad8689.zip"
USA_SOILGRIDS  = True
CONFINE_PATH   = None
OUT_DIR        = "."
SEED           = 42


## 0b. Classi di idoneità e gruppi forestali
Soglie e colori per le legende dei tre tipi di mappa (idoneità RF, probabilità Bayes, fasce del modello ibrido). Il dizionario `GRUPPI_FORESTALI` associa, tramite espressioni regolari, le descrizioni testuali dei tipi di bosco PAT a famiglie (peccete, faggete, lariceti, ecc.). Non esegue calcoli: definisce solo legende e categorie.

In [ ]:
CLASSI_RF = [
    (50, 61,  "#e6b800", "//",     "/",  "Idoneità 50–60%"),
    (61, 81,  "#ff7f0e", "\\\\",   "\\", "Idoneità 61–80%"),
    (81, 101, "#d62728", "xxx",    "x",  "Idoneità 81–100%"),
]

CLASSI_BAYES = [
    (2, 5,    "#e6b800", "//",     "/",  "Probabilità 2–5%"),
    (5, 10,   "#ff7f0e", "\\\\",   "\\", "Probabilità 5–10%"),
    (10, 100, "#8b0000", "xxx",    "x",  "Probabilità >10%"),
]

CLASSI_IBRIDE = [
    (30, 50,  "#e6b800", "//",     "/",  "Fascia Media"),
    (50, 80,  "#ff7f0e", "\\\\",   "\\", "Fascia Alta"),
    (80, 101, "#2b9348", "xxx",    "x",  "Hotspot Eccellente"),
]

GRUPPI_FORESTALI = {
    "peccete":       r"pecc|abete rosso|picea",
    "abetine":       r"abetin|abete bianco|abies",
    "faggete":       r"fagget|faggio|fagus",
    "lariceti_cembrete": r"larice|lariceto|cembro|cembret",
    "pinete":        r"pinet|pino|pinus|mugo",
    "latifoglie_termofile": r"querc|rovere|roverella|castagn|ostri|carpin|orniello|leccio|lecceta|robini",
    "altre_latifoglie": r"betul|acero|frassin|tiglio|ontan|alnet|salic|pioppo|latifoglie",
}


## 0c. Librerie e setup
Import delle librerie per geodati (`geopandas`, `rasterio`, `pyproj`, `shapely`), machine learning (`scikit-learn`), statistica bayesiana (`pymc`, `arviz`) e richieste HTTP. Vengono impostati i warning, la data odierna (usata per scegliere gli anni meteo da scaricare e i giorni da prevedere) e il generatore di numeri casuali.

In [ ]:
import os, glob, json, math, time, warnings
import numpy as np
import pandas as pd
import requests
import geopandas as gpd
import rasterio
from rasterio.transform import from_origin, rowcol
from rasterio.warp import reproject, Resampling
from rasterio.vrt import WarpedVRT
from rasterio.merge import merge
from rasterio.features import shapes, rasterize
from pyproj import Transformer
from shapely.geometry import shape, Polygon
from shapely.ops import unary_union
from numpy.lib.stride_tricks import sliding_window_view
from scipy.ndimage import gaussian_filter, distance_transform_edt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
import pymc as pm
import arviz as az

warnings.filterwarnings("ignore")
OGGI = pd.Timestamp.now(tz="Europe/Rome").tz_localize(None).normalize()
rng = np.random.default_rng(SEED)


## 1. Area di studio e griglia
Ricava il confine della Provincia di Trento (da file locale oppure da OpenStreetMap) e lo allinea a una griglia regolare da 100 m. La maschera `provincia` indica quali celle cadono dentro il territorio. Tutto il modello lavora su questa griglia: ogni cella è un'unità di previsione.

Le funzioni di supporto fanno tre cose: `su_griglia` riproietta qualsiasi raster sulla griglia del progetto, `riempi_nan` riempie i buchi nei dati con il valore del vicino più prossimo, `latlon_celle` converte gli indici di cella in coordinate geografiche.

In [ ]:
print("1/9 Confine provinciale e griglia...")
def carica_confine():
    if CONFINE_PATH:
        g = gpd.read_file(CONFINE_PATH).to_crs(CRS)
        return gpd.GeoDataFrame(geometry=[unary_union(list(g.geometry))], crs=CRS)
    r = requests.get(
        "https://nominatim.openstreetmap.org/search",
        params={"q": "Provincia di Trento", "countrycodes": "it", "format": "geojson", "polygon_geojson": 1, "limit": 10},
        headers={"User-Agent": "mappa-funghi-trentino/8.0"}, timeout=60)
    r.raise_for_status()
    feats = r.json().get("features", [])
    g = gpd.GeoDataFrame.from_features(feats, crs="EPSG:4326").to_crs(CRS)
    g = g[g.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    g["km2"] = g.area / 1e6
    g = g[(g.km2 > 5000) & (g.km2 < 7500)]
    return gpd.GeoDataFrame(geometry=[g.sort_values("km2", ascending=False).geometry.iloc[0]], crs=CRS)

confine = carica_confine()
minx, miny, maxx, maxy = confine.total_bounds
minx = math.floor(minx / RISOLUZIONE) * RISOLUZIONE
miny = math.floor(miny / RISOLUZIONE) * RISOLUZIONE
maxx = math.ceil(maxx / RISOLUZIONE) * RISOLUZIONE
maxy = math.ceil(maxy / RISOLUZIONE) * RISOLUZIONE
W = int((maxx - minx) / RISOLUZIONE)
H = int((maxy - miny) / RISOLUZIONE)
transform = from_origin(minx, maxy, RISOLUZIONE, RISOLUZIONE)
provincia = rasterize([(confine.geometry.iloc[0], 1)], out_shape=(H, W), transform=transform, fill=0, dtype="uint8").astype(bool)
lonmin, latmin, lonmax, latmax = confine.to_crs("EPSG:4326").total_bounds
tr_inv = Transformer.from_crs(CRS, "EPSG:4326", always_xy=True)
tr_dir = Transformer.from_crs("EPSG:4326", CRS, always_xy=True)

def su_griglia(src, resampling):
    with WarpedVRT(src, crs=CRS, transform=transform, width=W, height=H, resampling=resampling) as vrt:
        return vrt.read(1, masked=True).astype("float32").filled(np.nan)

def riempi_nan(a):
    m = np.isnan(a)
    if not m.any() or m.all(): return a
    idx = distance_transform_edt(m, return_distances=False, return_indices=True)
    return a[tuple(idx)]

def latlon_celle(idx):
    r, c = np.divmod(idx, W)
    x = minx + (c + 0.5) * RISOLUZIONE
    y = maxy - (r + 0.5) * RISOLUZIONE
    lon, lat = tr_inv.transform(x, y)
    return np.asarray(lat), np.asarray(lon)


## 2. Covariate statiche: terreno, bosco, suolo
Costruisce le variabili che non cambiano nel tempo:
- **Terreno**: quota (DEM Copernicus), pendenza, e due componenti dell'esposizione (nord-ità ed est-ità, derivate dai gradienti del DEM).
- **Bosco**: la maschera è l'intersezione tra ESA WorldCover (classe 10 = alberi) e i poligoni dei tipi forestali PAT. È l'universo in cui il fungo può trovarsi. I gruppi forestali PAT diventano covariate binarie.
- **Suolo** (SoilGrids): pH, sabbia e carbonio organico, se disponibili.

Ogni sorgente opzionale è protetta da `try/except`: se manca, il modello prosegue con le altre.

In [ ]:
print("2/9 Terreno, tipi di bosco, suolo...")
def carica_dem():
    dsets = []
    for lat in range(math.floor(latmin), math.floor(latmax) + 1):
        for lon in range(math.floor(lonmin), math.floor(lonmax) + 1):
            nome = f"Copernicus_DSM_COG_10_N{lat:02d}_00_E{lon:03d}_00_DEM"
            dsets.append(rasterio.open(f"/vsicurl/https://copernicus-dem-30m.s3.amazonaws.com/{nome}/{nome}.tif"))
    arr, tr = merge(dsets, bounds=(lonmin - 0.02, latmin - 0.02, lonmax + 0.02, latmax + 0.02))
    for d in dsets: d.close()
    dem_ = np.full((H, W), np.nan, dtype="float32")
    reproject(arr[0].astype("float32"), dem_, src_transform=tr, src_crs="EPSG:4326",
              dst_transform=transform, dst_crs=CRS, resampling=Resampling.average, dst_nodata=np.nan)
    return dem_

def leggi_vettoriale(sorgente):
    sorgente = str(sorgente)
    if sorgente.startswith("http"):
        percorso = os.path.join(OUT_DIR, os.path.basename(sorgente.split("?")[0]))
        if not os.path.exists(percorso):
            with requests.get(sorgente, stream=True, timeout=300) as r:
                r.raise_for_status()
                with open(percorso, "wb") as f:
                    for blocco in r.iter_content(1 << 20): f.write(blocco)
        sorgente = percorso
    gdf = gpd.read_file(("zip://" + sorgente) if sorgente.lower().endswith(".zip") else sorgente)
    if gdf.crs is None: gdf = gdf.set_crs("EPSG:25832")
    return gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].to_crs(CRS)

def rasterizza(geoms):
    return rasterize(((g, 1) for g in geoms), out_shape=(H, W), transform=transform, fill=0, dtype="uint8")

def carica_tipi_forestali():
    gdf = leggi_vettoriale(TIPI_FORESTALI)
    cols = [c for c in gdf.columns if c != "geometry" and gdf[c].dtype == object]
    testo = gdf[cols].fillna("").astype(str).agg(" ".join, axis=1).str.lower()
    bosco_ = rasterizza(gdf.geometry).astype(bool)
    gruppi = {}
    for nome, regex in GRUPPI_FORESTALI.items():
        m = testo.str.contains(regex, regex=True)
        if m.any(): gruppi["tf_" + nome] = rasterizza(gdf.geometry[m]).astype("float32")
    return bosco_, gruppi

def carica_worldcover():
    # mappa delle classi ESA WorldCover (10 = arbusti/alberi, 50 = urbano, 80 = acqua) sulla griglia del progetto
    lat0 = int(math.floor(latmin / 3) * 3)
    lon0 = int(math.floor(lonmin / 3) * 3)
    url = f"https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/ESA_WorldCover_10m_2021_v200_N{lat0:02d}E{lon0:03d}_Map.tif"
    with rasterio.open("/vsicurl/" + url) as src:
        return np.nan_to_num(su_griglia(src, Resampling.mode)).round().astype(np.int16)

with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR"):
    dem = riempi_nan(carica_dem())
    # WorldCover (copertura arborea) è sempre usato: esclude acque, aree urbane e altro non boscato
    classi_wc = carica_worldcover()
    bosco_wc = classi_wc == 10
    try:
        bosco, feat_bosco = carica_tipi_forestali()
        bosco = bosco & bosco_wc
    except Exception as e:
        print(f"   ATTENZIONE tipi forestali PAT non disponibili ({e}): bosco solo da WorldCover")
        bosco, feat_bosco = bosco_wc, {}
    feat_suolo = {}
    if USA_SOILGRIDS:
        for var, nome in [("phh2o", "suolo_ph"), ("sand", "suolo_sabbia"), ("soc", "suolo_carbonio")]:
            try:
                with rasterio.open(f"/vsicurl/https://files.isric.org/soilgrids/latest/data/{var}/{var}_5-15cm_mean.vrt") as src:
                    a = su_griglia(src, Resampling.bilinear)
                if not np.isnan(a).all(): feat_suolo[nome] = riempi_nan(a)
            except Exception as e: print(f"   ATTENZIONE suolo {var} non disponibile: {e}")

dy, dx = np.gradient(dem, RISOLUZIONE)
aspetto = np.arctan2(-dx, dy)
feat_statiche = {"quota": dem, "pendenza": np.degrees(np.arctan(np.hypot(dx, dy))),
                 "nordicita": np.cos(aspetto), "esposizione_est": np.sin(aspetto)}
feat_statiche.update(feat_bosco)
feat_statiche.update(feat_suolo)
nomi_statici = list(feat_statiche.keys())

valid = provincia & bosco & ~np.isin(classi_wc, [50, 80])
print("   Classi WorldCover presenti in provincia:", dict(zip(*np.unique(classi_wc[provincia], return_counts=True))))
print(f"   Celle nella maschera del bosco: {int(valid.sum())} ({valid.sum() / max(provincia.sum(), 1) * 100:.1f}% della provincia)")


### Controllo della maschera e matrice delle feature
Vengono esclusi acque, aree urbane e celle non boscate. Tre punti di controllo (laghi e centro città) devono risultare fuori maschera: è un test di sanità sui dati. Infine si costruisce `X_valid`, una riga per cella di bosco valida con tutte le covariate statiche.

In [ ]:
def controllo_punto(lat, lon, nome):
    x, y = tr_dir.transform(lon, lat)
    r, c = rowcol(transform, x, y)
    r, c = int(np.asarray(r).item()), int(np.asarray(c).item())
    print(f"   Controllo {nome}: classe WorldCover = {int(classi_wc[r, c])}, in maschera = {bool(valid[r, c])}")

controllo_punto(46.14, 10.93, "lago di Molveno (deve essere False)")
controllo_punto(46.05, 11.27, "lago di Caldonazzo (deve essere False)")
controllo_punto(46.07, 11.12, "centro di Trento (deve essere False)")
valid_idx = np.flatnonzero(valid.ravel())
X_valid = np.column_stack([feat_statiche[n].ravel()[valid_idx] for n in nomi_statici]).astype("float32")
dem_cella = dem.ravel()[valid_idx]
lat_cella, lon_cella = latlon_celle(valid_idx)


## 3. Osservazioni: presenze e pseudo-assenze
- **Presenze**: occorrenze GBIF delle sei specie target nel periodo e nell'area, filtrate per incertezza di coordinata (≤ 1 km), mese di stagione e appartenenza a una cella di bosco valida.
- **Background (pseudo-assenze)**: occorrenze di funghi in generale (taxonKey 5), che rappresentano dove la gente cerca funghi e quindi lo sforzo di raccolta, più celle casuali di bosco con date estratte dalla stessa distribuzione temporale delle presenze.

Il modello ha bisogno di un confronto: il dato GBIF registra solo dove un fungo è stato trovato, non dove non c'è. Le pseudo-assenze colmano questo vuoto.

In [ ]:
print("3/9 Osservazioni GBIF...")
wkt = f"POLYGON(({lonmin} {latmin},{lonmax} {latmin},{lonmax} {latmax},{lonmin} {latmax},{lonmin} {latmin}))"
PARAM_BASE = {"geometry": wkt, "hasCoordinate": "true", "hasGeospatialIssue": "false", "occurrenceStatus": "PRESENT", "year": f"{ANNO_MIN_GBIF},{OGGI.year}"}

def gbif_occ(extra, max_rec=None):
    out, offset = [], 0
    while True:
        p = dict(PARAM_BASE, **extra, limit=300, offset=offset)
        
        for tentativo in range(6):
            try:
                r = requests.get("https://api.gbif.org/v1/occurrence/search", params=p, timeout=120)
                if r.status_code == 429:
                    print(f"   [429 GBIF] Rallentamento richiesto, attesa di {10 * (tentativo + 1)} secondi...")
                    time.sleep(10 * (tentativo + 1))
                    continue
                r.raise_for_status()
                break
            except Exception as e:
                if tentativo == 5: raise e
                time.sleep(5 * (tentativo + 1))
        else:
            raise RuntimeError("GBIF limite 429 superato persistentemente.")

        j = r.json()
        out.extend(j.get("results", []))
        offset += 300
        if j.get("endOfRecords") or offset >= 99000 or (max_rec and len(out) >= max_rec): break
        time.sleep(0.5)  # Breve pausa di cortesia tra una pagina e l'altra
        
    return out[:max_rec] if max_rec else out

def eventi(records):
    vuoto = pd.DataFrame({"idx": [], "data": pd.to_datetime([]), "lat": [], "lon": []})
    righe = [{"lat": r.get("decimalLatitude"), "lon": r.get("decimalLongitude"), "unc": r.get("coordinateUncertaintyInMeters"),
              "y": r.get("year"), "m": r.get("month"), "d": r.get("day")} for r in records]
    df = pd.DataFrame(righe)
    if df.empty: return vuoto
    df = df.dropna(subset=["lat", "lon", "y", "m", "d"])
    df = df[df["unc"].isna() | (df["unc"] <= 1000)]
    df["data"] = pd.to_datetime(dict(year=df.y.astype(int), month=df.m.astype(int), day=df.d.astype(int)), errors="coerce")
    df = df.dropna(subset=["data"])
    df = df[df.data.dt.month.between(*MESI_STAGIONE)]
    if df.empty: return vuoto
    x, y = tr_dir.transform(df.lon.values, df.lat.values)
    r, c = rowcol(transform, x, y)
    r, c = np.asarray(r), np.asarray(c)
    ok = (r >= 0) & (r < H) & (c >= 0) & (c < W)
    ev = pd.DataFrame({"idx": r[ok] * W + c[ok], "data": df.data.values[ok], "lat": df.lat.values[ok], "lon": df.lon.values[ok]})
    return ev[valid.ravel()[ev.idx.values]].drop_duplicates(["idx", "data"]).reset_index(drop=True)

rec = []
for nome in SPECIE: rec += gbif_occ({"scientificName": nome})
ev_pres = eventi(rec)
ev_tg = eventi(gbif_occ({"taxonKey": 5}, max_rec=MAX_SFONDO))
chiavi_p = set(zip(ev_pres.idx, ev_pres.data))
ev_tg = ev_tg[[(i, d) not in chiavi_p for i, d in zip(ev_tg.idx, ev_tg.data)]].reset_index(drop=True)
pool_date = pd.concat([ev_tg.data, ev_pres.data]).values
n_r = max(len(ev_tg), 1000)
celle_r = rng.choice(valid_idx, n_r)
lat_r, lon_r = latlon_celle(celle_r)
ev_rand = pd.DataFrame({"idx": celle_r, "data": rng.choice(pool_date, n_r), "lat": lat_r, "lon": lon_r})
ev_bg = pd.concat([ev_tg, ev_rand], ignore_index=True)


## 4. Covariate meteorologiche
### 4a. Storico meteo
Carica le serie giornaliere di precipitazione, temperatura media e minima, umidità relativa su una griglia a 0.2°, dal file di cache 2017–2024. Se la cache non copre l'anno corrente, scarica l'integrazione recente da Open-Meteo. Le funzioni `open_meteo_richiesta` e `scarica_serie` gestiscono i limiti di velocità dell'API.

In [ ]:
print("5/9 Caricamento storico meteo (2017-2024 + integrazione dinamica corrente)...")
METEO_VARS = ["precipitation_sum", "temperature_2m_mean", "temperature_2m_min", "relative_humidity_2m_mean"]
NOMI_METEO = ["prec_3d", "prec_7d", "prec_14d", "prec_30d", "tmed_7d", "tmed_14d", "tmin_7d", "gg_ultima_pioggia", "rh_7d", "vpd_7d"]
URL_ARCH = "https://archive-api.open-meteo.com/v1/archive"
URL_PREV = "https://api.open-meteo.com/v1/forecast"

lats = np.round(np.arange(math.floor(latmin / PASSO_METEO) * PASSO_METEO, math.ceil(latmax / PASSO_METEO) * PASSO_METEO + PASSO_METEO / 2, PASSO_METEO), 4)
lons = np.round(np.arange(math.floor(lonmin / PASSO_METEO) * PASSO_METEO, math.ceil(lonmax / PASSO_METEO) * PASSO_METEO + PASSO_METEO / 2, PASSO_METEO), 4)
NLAT, NLON = len(lats), len(lons)
PTS_LAT, PTS_LON = np.repeat(lats, NLON), np.tile(lons, NLAT)
K = NLAT * NLON

def open_meteo_richiesta(url, la, lo, extra):
    params = {"latitude": ",".join(f"{x:.3f}" for x in la), "longitude": ",".join(f"{x:.3f}" for x in lo), "daily": ",".join(METEO_VARS), "timezone": "Europe/Rome"}
    params.update(extra)
    for tentativo in range(6):
        try:
            r = requests.get(url, params=params, timeout=120)
            if r.status_code == 429:
                time.sleep(15 * (tentativo + 1))
                continue
            r.raise_for_status()
            return r.json() if isinstance(r.json(), list) else [r.json()]
        except Exception as e:
            if tentativo == 5: raise e
            time.sleep(10)
    raise RuntimeError("Open-Meteo limite raggiunto.")

def scarica_serie(url, extra):
    per_var = {v: [] for v in METEO_VARS}
    elev, date_ = [], None
    for k0 in range(0, K, 15):
        for r in open_meteo_richiesta(url, PTS_LAT[k0:k0+15], PTS_LON[k0:k0+15], extra):
            date_ = pd.to_datetime(r["daily"]["time"])
            elev.append(r.get("elevation", np.nan))
            for v in METEO_VARS: per_var[v].append(np.array(r["daily"][v], dtype=float))
        time.sleep(2)
    return date_, {v: np.column_stack(per_var[v]) for v in METEO_VARS}, np.array(elev, float).reshape(NLAT, NLON)

def meteo_storico():
    cache_path = "fonti esterne statiche/storico meteo/meteo_2017_2024_storico.npz"
    if not os.path.exists(cache_path):
        raise FileNotFoundError(f"Manca il file di storico in {cache_path}")
    
    print("   Caricamento storico meteo (2017-2024) da file statico...")
    z = np.load(cache_path, allow_pickle=True)
    date_static = pd.to_datetime(z["date"])
    serie_static = {v: z[v] for v in METEO_VARS}
    elev = z["elev"]
    
    ultimo_anno_statico = date_static.year.max()
    if OGGI.year > ultimo_anno_statico:
        print(f"   Rilevati dati attuali post-{ultimo_anno_statico}. Scaricamento integrazione recente...")
        date_nov, serie_nov = [], {v: [] for v in METEO_VARS}
        for a in range(ultimo_anno_statico + 1, OGGI.year + 1):
            inizio = f"{a}-01-01"
            fine = min(pd.Timestamp(f"{a}-11-30"), OGGI - pd.Timedelta(days=1))
            if fine < pd.Timestamp(f"{a}-06-01"): continue
            d_n, s_n, _ = scarica_serie(URL_ARCH, {"start_date": inizio, "end_date": fine.strftime("%Y-%m-%d")})
            date_nov.append(d_n)
            for v in METEO_VARS: serie_nov[v].append(s_n[v])
            time.sleep(2)
        if date_nov:
            date_nov_concat = pd.DatetimeIndex(np.concatenate([d.values for d in date_nov]))
            serie_nov_concat = {v: np.vstack(serie_nov[v]) for v in METEO_VARS}
            date_static = date_static.append(date_nov_concat)
            for v in METEO_VARS:
                serie_static[v] = np.vstack([serie_static[v], serie_nov_concat[v]])
                
    return date_static, serie_static, elev


### 4b. Feature meteo
Dalle serie giornaliere ricava indicatori legati alla biologia del fungo:
- pioggia cumulata su 3, 7, 14 e 30 giorni;
- temperatura media su 7 e 14 giorni, e minima settimanale (gelate);
- giorni dall'ultima pioggia significativa (≥ 5 mm);
- umidità media e **VPD** (deficit di pressione di vapore), che misura quanto l'aria è secca.

Le feature sono calcolate anno per anno, così le finestre non attraversano due stagioni diverse.

In [ ]:
def feature_meteo(P, Tm, Tn, RH):
    T, Kk = P.shape
    def somma(a, n):
        cs = np.vstack([np.zeros((1, Kk)), np.cumsum(np.nan_to_num(a), axis=0)])
        out = np.full((T, Kk), np.nan, dtype="float32")
        out[n:] = cs[n:T] - cs[0:T - n]
        return out
    
    svp = 0.61078 * np.exp((17.27 * Tm) / (Tm + 237.3))
    vpd = svp * (1.0 - np.clip(RH, 0, 100) / 100.0)

    f = {
        "prec_3d": somma(P, 3), "prec_7d": somma(P, 7), "prec_14d": somma(P, 14), "prec_30d": somma(P, 30),
        "tmed_7d": somma(Tm, 7) / 7, "tmed_14d": somma(Tm, 14) / 14,
        "rh_7d": somma(RH, 7) / 7, "vpd_7d": somma(vpd, 7) / 7
    }
    mn = sliding_window_view(Tn, 7, axis=0).min(axis=-1)
    f["tmin_7d"] = np.full((T, Kk), np.nan, dtype="float32")
    f["tmin_7d"][7:] = mn[:T - 7]
    
    gg, cur = np.empty((T, Kk), dtype="float32"), np.full(Kk, 30.0)
    for t in range(T):
        gg[t] = cur
        cur = np.where(P[t] >= 5, 1.0, np.minimum(cur + 1, 30.0))
    f["gg_ultima_pioggia"] = gg
    
    return {n: f[n].astype("float32").reshape(T, NLAT, NLON) for n in NOMI_METEO}

def feature_meteo_stagioni(serie, date):
    parts = {n: [] for n in NOMI_METEO}
    for a in np.unique(date.year):
        m = np.asarray(date.year == a)
        f = feature_meteo(serie["precipitation_sum"][m], serie["temperature_2m_mean"][m],
                          serie["temperature_2m_min"][m], serie["relative_humidity_2m_mean"][m])
        for n in NOMI_METEO: parts[n].append(f[n])
    return {n: np.concatenate(parts[n], axis=0) for n in NOMI_METEO}

date_st, serie_st, elev_st = meteo_storico()
F_st = feature_meteo_stagioni(serie_st, date_st)


### 4c. Interpolazione e matrice degli eventi
`pesi_bilin` e `campiona` interpolano bilinearmente i valori meteo nella posizione precisa di ogni evento. `matrice_meteo` corregge la temperatura per la quota della cella (lapse rate). `matrice_eventi` compone la matrice finale per gli eventi, una riga per evento, con colonne: covariate statiche, meteo e giorno dell'anno. Gli eventi con dati mancanti vengono scartati.

In [ ]:
def pesi_bilin(lat, lon):
    fi = np.clip((np.asarray(lat) - lats[0]) / PASSO_METEO, 0, NLAT - 1.000001)
    fj = np.clip((np.asarray(lon) - lons[0]) / PASSO_METEO, 0, NLON - 1.000001)
    return fi.astype(int), fj.astype(int), fi - fi.astype(int), fj - fj.astype(int)

def campiona(F3, w, tt=0):
    i0, j0, di, dj = w
    return (F3[tt, i0, j0] * (1 - di) * (1 - dj) + F3[tt, i0 + 1, j0] * di * (1 - dj)
            + F3[tt, i0, j0 + 1] * (1 - di) * dj + F3[tt, i0 + 1, j0 + 1] * di * dj).astype("float32")

def matrice_meteo(F3, w, tt, quota, elev):
    dz = quota - campiona(elev[None], w, 0)
    cols = []
    for n in NOMI_METEO:
        v = campiona(F3[n], w, tt)
        if n.startswith(("tmed", "tmin")): v = v + LAPSE * dz
        cols.append(v)
    return np.column_stack(cols)

def matrice_eventi(ev):
    t = date_st.get_indexer(pd.DatetimeIndex(ev.data))
    ev = ev[t >= 0].reset_index(drop=True)
    t = t[t >= 0]
    M = matrice_meteo(F_st, pesi_bilin(ev.lat.values, ev.lon.values), t, dem.ravel()[ev.idx.values], elev_st)
    S = X_valid[np.searchsorted(valid_idx, ev.idx.values)]
    doy = ev.data.dt.dayofyear.values[:, None].astype("float32")
    X = np.hstack([S, M, doy])
    ok = np.isfinite(X).all(axis=1)
    return ev[ok].reset_index(drop=True), X[ok]

ev_p, Xp = matrice_eventi(ev_pres)
ev_b, Xb = matrice_eventi(ev_bg)


## 5. Addestramento e validazione
### 5a. Split e feature spaziale
Presenze e pseudo-assenze vengono divise in training (90%) e test (10%), stratificate per classe. Il test resta fuori da ogni scelta e misura la bontà su dati mai visti.

Viene inoltre creata la feature **trend di raccolta**: la densità delle presenze smussata con un filtro gaussiano (σ = 15 celle, circa 1.5 km). Per le presenze di training il trend è calcolato in modo *out-of-fold*: ogni presenza riceve un valore calcolato senza le proprie presenze, per evitare la fuga di informazione (leakage).

In [ ]:
print("6/9 Addestramento modelli con split 90/10 stratificato...")
X = np.vstack([Xp, Xb])
y = np.concatenate([np.ones(len(Xp)), np.zeros(len(Xb))])
n_p = len(Xp)
cell_pres = ev_p.idx.values
cell_all = np.concatenate([ev_p.idx.values, ev_b.idx.values])
RF_KW = dict(n_estimators=400, min_samples_leaf=3, max_features="sqrt",
             class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)

def griglia_trend(celle):
    g = np.zeros((H, W), dtype="float32")
    r_, c_ = np.divmod(celle, W)
    np.add.at(g, (r_, c_), 1.0)
    g = gaussian_filter(g, sigma=15.0)
    return ((g - g.min()) / (g.max() - g.min() + 1e-8)).astype("float32")

def trend_oof(celle, n_fold=5):
    # ogni presenza riceve il trend calcolato SENZA le presenze del proprio fold (niente leakage)
    out = np.zeros(len(celle), dtype="float32")
    fold = rng.integers(0, n_fold, len(celle))
    for f in range(n_fold):
        m = fold == f
        out[m] = griglia_trend(celle[~m]).ravel()[celle[m]]
    return out

idx_tutti = np.arange(len(y))
tr_i, te_i = train_test_split(idx_tutti, test_size=0.10, random_state=SEED, stratify=y)
X_train, X_test, y_train, y_test = X[tr_i], X[te_i], y[tr_i], y[te_i]

# feature spaziale per la valutazione: costruita solo con le presenze di training
tr_pres = tr_i[tr_i < n_p]
trend_eval = griglia_trend(cell_pres[tr_pres]).ravel()[cell_all].astype("float32")
trend_eval[tr_pres] = trend_oof(cell_pres[tr_pres])
X_spat = np.hstack([X, trend_eval[:, None]])
X_train_spat, X_test_spat = X_spat[tr_i], X_spat[te_i]


### 5b. Quattro modelli e metriche di test
Gli stessi dati di training alimentano quattro modelli:
- **RF** (Idoneità relativa): random forest, con l'errore out-of-bag come stima interna.
- **Bayes**: regressione logistica bayesiana stimata con MCMC (PyMC), che fornisce probabilità con la loro incertezza.
- **Spatial**: random forest che usa anche il trend di raccolta.
- **Stacking**: media 50/50 di RF e Bayes.

La soglia di decisione per le metriche è scelta sul training con l'indice di Youden e applicata al test. Le metriche riportate sono AUC, accuratezza, sensibilità, specificità e precisione.

In [ ]:
def soglia_youden(probs, y_true):
    fpr, tpr, th = roc_curve(y_true, probs)
    return th[np.argmax(tpr - fpr)]

def calcola_metriche(p_tr, y_tr, p_te, y_te):
    # soglia scelta sul training (OOB per le foreste), poi riportata sul test
    s_ = soglia_youden(p_tr, y_tr)
    y_pred = (p_te >= s_).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred, labels=[0, 1]).ravel()
    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    sens = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    prec = tp / max(tp + fp, 1)
    auc = roc_auc_score(y_te, p_te)
    return {"auc": round(float(auc), 3), "accuratezza": round(float(acc * 100), 1),
            "sensibilita": round(float(sens * 100), 1), "specificita": round(float(spec * 100), 1),
            "precisione": round(float(prec * 100), 1)}

rf_model = RandomForestClassifier(oob_score=True, **RF_KW).fit(X_train, y_train)
p_rf_tr = np.nan_to_num(rf_model.oob_decision_function_[:, 1])
p_rf_te = rf_model.predict_proba(X_test)[:, 1]
metriche_rf = calcola_metriche(p_rf_tr, y_train, p_rf_te, y_test)

X_mean, X_std = X_train.mean(axis=0), X_train.std(axis=0) + 1e-8
X_train_scaled, X_test_scaled = (X_train - X_mean) / X_std, (X_test - X_mean) / X_std

with pm.Model() as bayesian_model:
    alpha = pm.Normal("alpha", mu=0, sigma=2)
    beta = pm.Normal("beta", mu=0, sigma=2, shape=X_train_scaled.shape[1])
    mu_lin = alpha + pm.math.dot(X_train_scaled, beta)
    y_obs = pm.Bernoulli("y_obs", p=pm.math.sigmoid(mu_lin), observed=y_train)
    trace = pm.sample(draws=800, tune=800, chains=2, target_accept=0.9, random_seed=SEED, return_inferencedata=True)

summ = az.summary(trace, var_names=["alpha", "beta"])
print(f"   Bayes convergenza: R-hat max {summ['r_hat'].max():.3f}, ESS bulk min {summ['ess_bulk'].min():.0f}")
alpha_mean = float(trace.posterior["alpha"].values.mean())
beta_mean = trace.posterior["beta"].values.mean(axis=(0, 1))
sigm = lambda z: 1 / (1 + np.exp(-z))
p_bay_tr = sigm(alpha_mean + X_train_scaled @ beta_mean)
p_bay_te = sigm(alpha_mean + X_test_scaled @ beta_mean)
metriche_bayes = calcola_metriche(p_bay_tr, y_train, p_bay_te, y_test)

rf_spatial = RandomForestClassifier(oob_score=True, **RF_KW).fit(X_train_spat, y_train)
p_sp_tr = np.nan_to_num(rf_spatial.oob_decision_function_[:, 1])
p_sp_te = rf_spatial.predict_proba(X_test_spat)[:, 1]
metriche_spatial = calcola_metriche(p_sp_tr, y_train, p_sp_te, y_test)

# stacking valutato davvero (prima riportava le metriche dell'RF)
p_st_tr = 0.5 * (p_rf_tr + p_bay_tr)
p_st_te = 0.5 * (p_rf_te + p_bay_te)
metriche_stacking = calcola_metriche(p_st_tr, y_train, p_st_te, y_test)


### 5c. Modelli finali
Random forest e random forest spaziale vengono riaddestrati su tutti i dati, per le previsioni operative. Il modello Bayes non viene rifatto: usa il valore medio a posteriori già stimato (`alpha_mean`, `beta_mean`).

In [ ]:
# modelli finali su tutti i dati
rf_model_full = RandomForestClassifier(**RF_KW).fit(X, y)
grid_full = griglia_trend(cell_pres)
trend_full = np.concatenate([trend_oof(cell_pres), grid_full.ravel()[ev_b.idx.values]])
X_spat_full = np.hstack([X, trend_full[:, None]])
rf_spatial_full = RandomForestClassifier(**RF_KW).fit(X_spat_full, y)
trend_spaziale_grid = grid_full
trend_valid = grid_full.ravel()[valid_idx][:, None]


## 6. Previsione per oggi, domani e dopodomani
Scarica il meteo degli ultimi 35 giorni più le previsioni di 3 giorni. Per ciascun giorno costruisce le feature di ogni cella di bosco valida e applica i modelli finali:
- **RF** e **Spatial**: probabilità normalizzate sul 99° percentile e riportate su scala 0–100;
- **Bayes**: probabilità × 100 (scala 0–100 diretta);
- **Stacking**: media 50/50 tra la versione normalizzata di RF e quella di Bayes.

In [ ]:
print("7/9 Previsioni meteo aggiornate per i 3 giorni target...")
def open_meteo_prev(url, la, lo):
    params = {"latitude": ",".join(f"{x:.3f}" for x in la), "longitude": ",".join(f"{x:.3f}" for x in lo), "daily": ",".join(METEO_VARS), "timezone": "Europe/Rome", "past_days": 35, "forecast_days": 3}
    r = requests.get(url, params=params, timeout=180)
    r.raise_for_status()
    res = r.json() if isinstance(r.json(), list) else [r.json()]
    elev, per_var, date_ = [], {v: [] for v in METEO_VARS}, None
    for item in res:
        date_ = pd.to_datetime(item["daily"]["time"])
        elev.append(item.get("elevation", np.nan))
        for v in METEO_VARS: per_var[v].append(np.array(item["daily"][v], dtype=float))
    return date_, {v: np.column_stack(per_var[v]) for v in METEO_VARS}, np.array(elev, float).reshape(len(lats), len(lons))

PTS_LAT_B, PTS_LON_B = np.repeat(lats, NLON), np.tile(lons, NLAT)
date_pr, serie_pr, elev_pr = open_meteo_prev(URL_PREV, PTS_LAT_B, PTS_LON_B)
F_pr = feature_meteo(serie_pr["precipitation_sum"], serie_pr["temperature_2m_mean"], serie_pr["temperature_2m_min"], serie_pr["relative_humidity_2m_mean"])
w_cella = pesi_bilin(lat_cella, lon_cella)

ETICHETTE = ["Oggi", "Domani (+24 h)", "Dopodomani (+48 h)"]
date_giorni, rf_giorni, bayes_giorni, spatial_giorni, stacking_giorni = [], [], [], [], []

for d in range(3):
    giorno = OGGI + pd.Timedelta(days=d)
    t = date_pr.get_loc(giorno)
    M = matrice_meteo(F_pr, w_cella, t, dem_cella, elev_pr)
    M = np.where(np.isfinite(M), M, np.nanmedian(M, axis=0))
    Xd = np.hstack([X_valid, M, np.full((len(valid_idx), 1), giorno.dayofyear, dtype="float32")])
    
    p_rf = np.concatenate([rf_model_full.predict_proba(Xd[i:i + 200000])[:, 1] for i in range(0, len(Xd), 200000)])
    rf_giorni.append(np.clip(p_rf / max(np.percentile(p_rf, 99), 1e-6) * 100, 0, 100))
    
    sc_bayes = (1 / (1 + np.exp(-(alpha_mean + np.dot((Xd - X_mean) / X_std, beta_mean))))) * 100.0
    bayes_giorni.append(sc_bayes)
    
    p_spat = np.concatenate([rf_spatial_full.predict_proba(np.hstack([Xd, trend_valid])[i:i + 200000])[:, 1] for i in range(0, len(Xd), 200000)])
    spatial_giorni.append(np.clip(p_spat / max(np.percentile(p_spat, 99), 1e-6) * 100, 0, 100))
    
    sc_bayes_norm = np.clip(sc_bayes / max(np.percentile(sc_bayes, 99), 1e-6) * 100, 0, 100)
    stacking_giorni.append(0.5 * rf_giorni[-1] + 0.5 * sc_bayes_norm)
    date_giorni.append(giorno)


## 7. Poligoni di idoneità
### 7a. Dai punteggi alle aree
`genera_poligoni` trasforma i punteggi per cella in aree in quattro passi:
1. smussatura gaussiana (150 m), limitata alle celle di bosco;
2. classificazione nelle fasce definite in `CLASSI_*`;
3. conversione da raster a poligoni, con rimozione dei frammenti sotto 0.01 km²;
4. semplificazione geometrica per alleggerire il file finale.

Restituisce per ogni modello e per ogni giorno i poligoni e l'area in km² di ciascuna fascia.

In [ ]:
def genera_poligoni(score_list, classi_config):
    giorni_poligoni, giorni_km2 = [], []
    for d in range(3):
        g = np.full(H * W, np.nan, dtype="float32")
        g[valid_idx] = score_list[d]
        g = g.reshape(H, W)
        if SMOOTH_M > 0:
            s = SMOOTH_M / RISOLUZIONE
            num = gaussian_filter(np.nan_to_num(g) * valid, s)
            den = gaussian_filter(valid.astype("float32"), s)
            g = np.where(valid & (den > 0.05), num / np.maximum(den, 1e-6), np.nan).astype("float32")
        cls = np.zeros((H, W), dtype="uint8")
        for k, (lo, hi, *_r) in enumerate(classi_config, 1):
            cls[(g >= lo) & (g < hi)] = k
        out_pol, km2_list = [], []
        for k, (lo, hi, col, hat, tipo, lab) in enumerate(classi_config, 1):
            m = (cls == k)
            geoms = [shape(geom) for geom, v in shapes(m.astype("uint8"), mask=m, transform=transform) if v == 1]
            if not geoms:
                out_pol.append(gpd.GeoDataFrame(geometry=[], crs=CRS))
                km2_list.append(0.0)
                continue
            poly_gdf = gpd.GeoDataFrame(geometry=[unary_union(geoms)], crs=CRS).explode(index_parts=False)
            poly_gdf = poly_gdf[poly_gdf.area / 1e6 >= MIN_AREA_KM2].copy()
            km2_list.append(float((cls == k).sum() * RISOLUZIONE**2 / 1e6))
            if not poly_gdf.empty:
                poly_gdf["geometry"] = poly_gdf.simplify(RISOLUZIONE * 0.5)
                poly_gdf["classe"], poly_gdf["colore"], poly_gdf["tratteggio"], poly_gdf["tipo"] = lab, col, hat, tipo
            out_pol.append(poly_gdf)
        giorni_poligoni.append(pd.concat(out_pol, ignore_index=True) if out_pol else None)
        giorni_km2.append(km2_list)
    return giorni_poligoni, giorni_km2

poly_rf, km2_rf = genera_poligoni(rf_giorni, CLASSI_RF)
poly_bayes, km2_bayes = genera_poligoni(bayes_giorni, CLASSI_BAYES)
poly_spatial, km2_spatial = genera_poligoni(spatial_giorni, CLASSI_IBRIDE)
poly_stacking, km2_stacking = genera_poligoni(stacking_giorni, CLASSI_IBRIDE)


### 7b. Sentieri SAT
Legge i file GPX presenti nella cartella dei sentieri e li unisce in un unico layer, riproiettato su EPSG:25832. Se non ci sono file, il layer resta vuoto e la mappa funziona lo stesso.

In [ ]:
gpx_files = glob.glob("fonti esterne statiche/sentieri/*.gpx")
cai_gdf = None
if gpx_files:
    gpx_list = []
    for gf in gpx_files:
        try:
            temp_gpx = gpd.read_file(gf, layer='tracks')
            if temp_gpx.empty: temp_gpx = gpd.read_file(gf)
            gpx_list.append(temp_gpx.to_crs(CRS))
        except Exception: pass
    if gpx_list:
        cai_gdf = gpd.GeoDataFrame(pd.concat(gpx_list, ignore_index=True), crs=CRS)


## 8. Dati per la mappa (GeoJSON)
`a_geojson` converte i layer in GeoJSON WGS84 con coordinate arrotondate a 5 decimali (circa un metro). `maschera_esterna` è un poligono che copre il mondo meno la provincia, così la parte fuori confine viene oscurata. Il dizionario `dati_web` raccoglie tutto: bounds, confine, sentieri e, per ciascun modello, metriche, legenda e poligoni dei tre giorni. È l'unico oggetto che passa dal Python al JavaScript.

In [ ]:
def arrotonda(c, nd=5):
    return round(c, nd) if isinstance(c, (int, float)) else [arrotonda(x, nd) for x in c]

def a_geojson(gdf):
    feats = []
    if gdf is not None and not gdf.empty:
        gdf_wgs = gdf.to_crs("EPSG:4326") if gdf.crs != "EPSG:4326" else gdf
        for geom in gdf_wgs.geometry:
            if geom.is_empty: continue
            gi_inf = geom.__geo_interface__
            feats.append({"type": "Feature", "properties": {}, "geometry": {"type": gi_inf["type"], "coordinates": arrotonda(gi_inf["coordinates"])}})
    return {"type": "FeatureCollection", "features": feats}

world_box = Polygon([(-180, -90), (180, -90), (180, 90), (-180, 90), (-180, -90)])
trentino_wgs = unary_union(list(confine.to_crs("EPSG:4326").geometry))
mask_poly = world_box.difference(trentino_wgs)
mask_geojson = {"type": "FeatureCollection", "features": [{"type": "Feature", "geometry": mask_poly.__geo_interface__, "properties": {}}]}

dati_web = {
    "bounds": [[float(latmin), float(lonmin)], [float(latmax), float(lonmax)]],
    "generato": OGGI.strftime("%d/%m/%Y"),
    "confine": a_geojson(confine.to_crs("EPSG:4326")),
    "maschera_esterna": mask_geojson,
    "cai": a_geojson(cai_gdf) if cai_gdf is not None else {"type": "FeatureCollection", "features": []},
    "modelli": {
        "rf": {"titolo": "Idoneità Relativa (RF)", "metriche": metriche_rf, "classi": [{"etichetta": c[5], "colore": c[2], "tipo": c[4]} for c in CLASSI_RF],
               "giorni": [{"etichetta": ETICHETTE[d], "data": date_giorni[d].strftime("%d/%m"), "km2": km2_rf[d],
                           "classi": [a_geojson(poly_rf[d][poly_rf[d].classe == c[5]]) if poly_rf[d] is not None and not poly_rf[d].empty else {"type": "FeatureCollection", "features": []} for c in CLASSI_RF]} for d in range(3)]},
        "bayes": {"titolo": "Probabilità stimata (Bayes)", "metriche": metriche_bayes, "classi": [{"etichetta": c[5], "colore": c[2], "tipo": c[4]} for c in CLASSI_BAYES],
                  "giorni": [{"etichetta": ETICHETTE[d], "data": date_giorni[d].strftime("%d/%m"), "km2": km2_bayes[d],
                              "classi": [a_geojson(poly_bayes[d][poly_bayes[d].classe == c[5]]) if poly_bayes[d] is not None and not poly_bayes[d].empty else {"type": "FeatureCollection", "features": []} for c in CLASSI_BAYES]} for d in range(3)]},
        "spatial": {"titolo": "Spatial Machine Learning", "metriche": metriche_spatial, "classi": [{"etichetta": c[5], "colore": c[2], "tipo": c[4]} for c in CLASSI_IBRIDE],
                    "giorni": [{"etichetta": ETICHETTE[d], "data": date_giorni[d].strftime("%d/%m"), "km2": km2_spatial[d],
                                "classi": [a_geojson(poly_spatial[d][poly_spatial[d].classe == c[5]]) if poly_spatial[d] is not None and not poly_spatial[d].empty else {"type": "FeatureCollection", "features": []} for c in CLASSI_IBRIDE]} for d in range(3)]},
        "stacking": {"titolo": "Ensemble Stacking", "metriche": metriche_stacking, "classi": [{"etichetta": c[5], "colore": c[2], "tipo": c[4]} for c in CLASSI_IBRIDE],
                     "giorni": [{"etichetta": ETICHETTE[d], "data": date_giorni[d].strftime("%d/%m"), "km2": km2_stacking[d],
                                 "classi": [a_geojson(poly_stacking[d][poly_stacking[d].classe == c[5]]) if poly_stacking[d] is not None and not poly_stacking[d].empty else {"type": "FeatureCollection", "features": []} for c in CLASSI_IBRIDE]} for d in range(3)]}
    }
}


## 9. Template HTML della mappa
Stringa HTML con Leaflet, che contiene:
- layer topografico e satellitare, con il confine e l'oscuramento esterno;
- pannello di controllo: scelta del modello, metriche, selezione del giorno, legenda e attivazione dei sentieri SAT;
- retini SVG per distinguere le classi anche in bianco e nero;
- pulsante GPS con bussola (su iOS il permesso viene richiesto dentro il click dell'utente).

Il segnaposto `__DATI__` verrà sostituito con il JSON della cella successiva.

In [ ]:
HTML_CONTENT = """<!DOCTYPE html>
<html lang="it"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0, viewport-fit=cover">
<title>Sottobosco - Trentino</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
 html,body,#map{height:100%;margin:0;padding:0;font-family:system-ui,sans-serif;}
 .pannello{background:rgba(255,255,255,.97);padding:10px 12px;border-radius:10px;font-size:13px;line-height:1.4;
           max-width:300px;width:calc(100vw - 40px);box-shadow:0 4px 15px rgba(0,0,0,.3);transition:all 0.3s ease;}
 .pannello.compresso{max-height:35px;overflow:hidden;padding:6px 12px;}
 .btn-toggle{background:none;border:0;font-weight:bold;cursor:pointer;float:right;font-size:14px;color:#2d6a4f;}
 .pannello select, .pannello button{display:block;width:100%;margin:5px 0;padding:6px;border:1px solid #888;border-radius:6px;background:#f8f9fa;cursor:pointer;font:inherit}
 .pannello button.attivo{background:#2d6a4f;color:#fff;border-color:#2d6a4f}
 .riga{display:flex;align-items:center;gap:8px;margin:3px 0}
 .box-metriche{background:#eef2f3;padding:6px;border-radius:5px;margin-top:5px;font-size:11px;}
 hr{border:0;border-top:1px solid #ccc;margin:6px 0}
 small{color:#555}
 .btn-gps{background:#fff;border:2px solid rgba(0,0,0,0.2);border-radius:6px;width:36px;height:36px;line-height:32px;text-align:center;font-size:18px;cursor:pointer;box-shadow:0 1px 5px rgba(0,0,0,0.4);display:block;text-decoration:none;color:#000;}
.btn-gps.attivo{background:#1d4ed8;color:#fff;border-color:#1d4ed8}
</style></head><body><div id="map"></div>
<script>
const D = __DATI__;
const map = L.map('map', {zoomControl: false}).fitBounds(D.bounds);
L.control.zoom({position: 'bottomright'}).addTo(map);

const opentopo = L.tileLayer('https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png', {
  maxZoom: 17, attribution: 'Map data: © OpenStreetMap contributors, SRTM | Map style: © OpenTopoMap'
});
const sat = L.tileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}', {maxZoom:18, attribution: 'Tiles © Esri — Source: Esri, Maxar, Earthstar Geographics, GIS User Community'});
opentopo.addTo(map);
L.control.layers({'OpenTopoMap': opentopo, 'Satellite': sat}, null, {position:'topright'}).addTo(map);

L.geoJSON(D.maschera_esterna, {style:{color: '#000', weight: 0, fillColor: '#000', fillOpacity: 0.75}, interactive: false}).addTo(map);
L.geoJSON(D.confine, {style:{color:'#000', weight:2, fill:false}, interactive:false}).addTo(map);

const caiLayer = L.geoJSON(D.cai, {style:{color:'#d90429', weight:2, opacity:0.8}});

const svg = L.svg({padding:0.5}).addTo(map);
const NS = 'http://www.w3.org/2000/svg';
const defs = document.createElementNS(NS, 'defs');

function creaPattern(classi) {
  defs.innerHTML = '';
  classi.forEach((c, i) => {
    const p = document.createElementNS(NS, 'pattern');
    p.setAttribute('id', 'hatch' + i); p.setAttribute('patternUnits', 'userSpaceOnUse');
    p.setAttribute('width', 8); p.setAttribute('height', 8);
    p.setAttribute('patternTransform', 'rotate(' + (c.tipo === '\\\\' ? -45 : 45) + ')');
    const bg = document.createElementNS(NS, 'rect');
    bg.setAttribute('width', 8); bg.setAttribute('height', 8); bg.setAttribute('fill', c.colore); bg.setAttribute('fill-opacity', 0.25);
    p.appendChild(bg);
    const linea = d => { const e = document.createElementNS(NS, 'path'); e.setAttribute('d', d);
       e.setAttribute('stroke', c.colore); e.setAttribute('stroke-width', 1.8); e.setAttribute('fill', 'none'); p.appendChild(e); };
    linea('M4 0 L4 8');
    if (c.tipo === 'x') linea('M0 4 L8 4');
    defs.appendChild(p);
  });
}
svg._container.appendChild(defs);

let currentModelKey = 'rf';
let currentDayIdx = 0;
let layerGroupAttivo = null;

function aggiornaMappa() {
  if (layerGroupAttivo) map.removeLayer(layerGroupAttivo);
  const mod = D.modelli[currentModelKey];
  creaPattern(mod.classi);
  
  layerGroupAttivo = L.layerGroup();
  const giornoData = mod.giorni[currentDayIdx];
  
  giornoData.classi.forEach((cl, i) => {
    if (!cl.features.length) return;
    L.geoJSON(cl, {renderer:svg, interactive:false,
      style:{color:mod.classi[i].colore, weight:1, opacity:.9, fillColor:'url(#hatch' + i + ')', fillOpacity:1}}).addTo(layerGroupAttivo);
  });
  layerGroupAttivo.addTo(map);

  const met = mod.metriche;
  document.getElementById('box-met').innerHTML = 
    `<b>Bontà (Test 10%):</b> AUC:<b>${met.auc}</b> | Acc:<b>${met.accuratezza}%</b> | Sens:<b>${met.sensibilita}%</b> | Spec:<span style="color:#d90429">${met.specificita}%</span>`;

  document.getElementById('leg').innerHTML = mod.classi.map((c, k) =>
    '<div class="riga"><svg width="22" height="14"><rect width="22" height="14" fill="url(#hatch' + k + ')" stroke="' + c.colore + '"/></svg>' +
    c.etichetta + ' <small>(' + giornoData.km2[k].toFixed(0) + ' km²)</small></div>').join('');
    
  document.querySelectorAll('#btns-giorni button').forEach((b, k) => b.classList.toggle('attivo', k === currentDayIdx));
}

const pannello = L.control({position:'topleft'});
pannello.onAdd = () => {
  const d = L.DomUtil.create('div', 'pannello');
  d.innerHTML = '<b>Sottobosco Trentino</b><button class="btn-toggle" id="btn-compri">−</button><br><small>' + D.generato + '</small>' +
                '<div id="contenuto-pannello">' +
                '<hr><b>Modello:</b>' +
                '<select id="sel-modello">' +
                  '<option value="rf">1. Idoneità Relativa (RF)</option>' +
                  '<option value="bayes">2. Probabilità stimata (Bayes)</option>' +
                  '<option value="spatial">3. Spatial Machine Learning</option>' +
                  '<option value="stacking">4. Ensemble Stacking</option>' +
                '</select>' +
                '<div id="box-met" class="box-metriche"></div>' +
                '<hr><b>Previsione:</b><div id="btns-giorni" style="display:flex;gap:4px"></div>' +
                '<hr><label><input type="checkbox" id="chk-cai"> Mostra Rete Sentieri SAT</label>' +
                '<hr><div id="leg"></div></div>';
  L.DomEvent.disableClickPropagation(d);
  return d;
};
pannello.addTo(map);

document.getElementById('btn-compri').onclick = () => {
  const pan = document.querySelector('.pannello');
  const btn = document.getElementById('btn-compri');
  pan.classList.toggle('compresso');
  btn.textContent = pan.classList.contains('compresso') ? '+' : '−';
};

document.getElementById('sel-modello').onchange = (e) => {
  currentModelKey = e.target.value;
  aggiornaMappa();
};

document.getElementById('chk-cai').onchange = (e) => {
  if(e.target.checked) caiLayer.addTo(map);
  else map.removeLayer(caiLayer);
};

const containerGiorni = document.getElementById('btns-giorni');
D.modelli.rf.giorni.forEach((g, i) => {
  const b = document.createElement('button');
  b.textContent = g.etichetta;
  b.style.flex = "1";
  b.style.fontSize = "11px";
  b.style.padding = "4px";
  b.onclick = () => { currentDayIdx = i; aggiornaMappa(); };
  containerGiorni.appendChild(b);
});

const iconaGps = L.divIcon({className:'', iconSize:[40,40], iconAnchor:[20,20],
  html:'<div id="freccia-gps" style="width:40px;height:40px;transition:transform .2s"><svg viewBox="0 0 40 40" width="40" height="40"><path d="M20 0 L28 17 L20 13 L12 17 Z" fill="#1d4ed8" fill-opacity="0.85" stroke="#fff" stroke-width="1.2"/><circle cx="20" cy="20" r="8" fill="#1d4ed8" stroke="#fff" stroke-width="3"/></svg></div>'});

const gpsControl = L.Control.extend({
  options: { position: 'bottomright' },
  onAdd: function () {
    const btn = L.DomUtil.create('a', 'btn-gps');
    btn.innerHTML = '📍';
    btn.title = 'Mostra la mia posizione in tempo reale';
    let attivo = false, watchId = null, marker = null, cerchio = null, primaFix = true, bussolaAttiva = false;

    function ruota(gradi) {
      const f = document.getElementById('freccia-gps');
      if (f && typeof gradi === 'number' && !isNaN(gradi)) f.style.transform = 'rotate(' + gradi + 'deg)';
    }
    function alOrientamento(e) {
      let h = null;
      if (typeof e.webkitCompassHeading === 'number') h = e.webkitCompassHeading;
      else if (e.absolute && typeof e.alpha === 'number') h = (360 - e.alpha) % 360;
      if (h !== null) { bussolaAttiva = true; ruota(h); }
    }
    function attivaBussola() {
      // va chiamata dentro il click dell'utente: iOS 13+ chiede il permesso esplicitamente
      if (typeof DeviceOrientationEvent !== 'undefined' && typeof DeviceOrientationEvent.requestPermission === 'function') {
        DeviceOrientationEvent.requestPermission()
          .then(function (r) { if (r === 'granted') window.addEventListener('deviceorientation', alOrientamento, true); })
          .catch(function () {});
      } else if ('ondeviceorientationabsolute' in window) {
        window.addEventListener('deviceorientationabsolute', alOrientamento, true);
      }
    }
    function alPosizione(pos) {
      const ll = [pos.coords.latitude, pos.coords.longitude];
      const prec = pos.coords.accuracy;
      if (!marker) {
        marker = L.marker(ll, {icon: iconaGps, interactive: false}).addTo(map);
        cerchio = L.circle(ll, {radius: prec, color: '#1d4ed8', weight: 1, fillOpacity: 0.1, interactive: false}).addTo(map);
      } else {
        marker.setLatLng(ll);
        cerchio.setLatLng(ll).setRadius(prec);
      }
      if (primaFix) { map.setView(ll, Math.max(map.getZoom(), 15)); primaFix = false; }
      else if (!map.getBounds().contains(ll)) map.panTo(ll);
      // senza bussola: direzione di movimento dal GPS
      if (!bussolaAttiva && typeof pos.coords.heading === 'number' && !isNaN(pos.coords.heading) && (pos.coords.speed || 0) > 0.5) ruota(pos.coords.heading);
    }
    function alErrore(e) {
      if (e.code === 3) return; // timeout: il watch continua
      const motivi = {1: 'permesso negato', 2: 'posizione non disponibile'};
      alert('Impossibile rilevare la posizione GPS: ' + (motivi[e.code] || e.message));
      spegni();
    }
    function spegni() {
      if (watchId !== null) navigator.geolocation.clearWatch(watchId);
      watchId = null; attivo = false; btn.classList.remove('attivo');
      if (marker) { map.removeLayer(marker); marker = null; }
      if (cerchio) { map.removeLayer(cerchio); cerchio = null; }
      primaFix = true;
    }
    function accendi() {
      if (!('geolocation' in navigator)) { alert('Geolocalizzazione non disponibile su questo browser.'); return; }
      attivo = true; btn.classList.add('attivo');
      attivaBussola();
      watchId = navigator.geolocation.watchPosition(alPosizione, alErrore, {enableHighAccuracy: true, maximumAge: 3000, timeout: 20000});
    }
    L.DomEvent.on(btn, 'click', function (e) {
      L.DomEvent.stopPropagation(e); L.DomEvent.preventDefault(e);
      if (attivo) spegni(); else accendi();
    });
    return btn;
  }
});
map.addControl(new gpsControl());

if (window.matchMedia('(max-width: 700px)').matches) {
  document.querySelector('.pannello').classList.add('compresso');
  document.getElementById('btn-compri').textContent = '+';
}
aggiornaMappa();
</script></body></html>"""


## 10. Scrittura di index.html
Sostituisce `__DATI__` con il JSON dei risultati e salva la pagina. Il file risultante è autonomo: si apre nel browser o si pubblica su un hosting statico.

In [ ]:
f_html = os.path.join(OUT_DIR, "index.html")
with open(f_html, "w", encoding="utf-8") as f:
    f.write(HTML_CONTENT.replace("__DATI__", json.dumps(dati_web, ensure_ascii=False)))

print(f"\nFatto! File aggiornato salvato come: {f_html}")